# Deney 9 - **D3**: varlik sadece zemin katta okunur

`MASK_KEY=1  MASK_BLK=1,2,3,4,5,6,7`

Onceden kayit: **ONKAYIT_D3_ZEMIN_KAT.md** (karar kurali kosudan once yazildi).

| kol | maske | durum |
|---|---|---|
| A | yok | var (deney4) |
| D | poz1 @ blok 6,7 | var (deney7) |
| **D3** | **poz1 @ blok 1..7** | **bu defter** |

Birincil olcu: 60.000-80.000 penceresi, 5 nokta agirlik ortalamasi, ENT.
Esdegerlik bandi +/-0.05.

**Hucre sirasi kasitli:** olcum hatti, D3 baslamadan ONCE, D'nin BILINEN
egrisine karsi dogrulanir (Hucre 2). Tutmazsa 85 dakika harcanmaz.


In [ ]:
# ============================================================================
# HUCRE 1 - KURULUM + UC DENETIM KAPISI
# ============================================================================
import subprocess, sys, os, hashlib, shutil, inspect, glob, re, json, time
import numpy as np, torch

assert torch.cuda.is_available(), "GPU YOK -> Runtime > Change runtime type > T4"
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

DR  = "/content/drive/MyDrive"
EV  = f"{DR}/deney9_D3"        # bu deneyin kalici klasoru
OUT = "/content/out9"          # calisma
KOD = "/content/_d3"           # klon
REF = {"D":  (f"{DR}/deney7_maske/D",          (6, 7)),      # referans kollar
       "A":  (f"{DR}/deney4_tekrarli_erisim",  ())}
MASKE_D3 = (1, 2, 3, 4, 5, 6, 7)      # TEK YERDE tanimli, elle kopyalanmaz
os.makedirs(f"{EV}/kod", exist_ok=True); os.makedirs(OUT, exist_ok=True)

subprocess.run(["bash", "-lc", f"""
pkill -f '[k]os9.py' ; sleep 1
cp -ru {EV}/. {OUT}/ 2>/dev/null
rm -rf {KOD} ; git clone -q https://github.com/sekerahmet/sekerai.git {KOD}
"""])
print("klon:", subprocess.run(["bash", "-lc", f"cd {KOD} && git log --oneline -1"],
                              capture_output=True, text=True).stdout.strip())

# DEPODAKI SURUM YAMASIZ (65381 char). Yamali surum yalnizca Drive'da (67224).
# Kaynak SIRAYLA aranir; bulunamazsa acik mesajla durulur.
_adaylar = [f"{EV}/kod/sifirdan.py",
            f"{DR}/deney8_tohum1/kod/sifirdan.py",
            f"{DR}/deney7_maske/kod/sifirdan.py"]
src = next((p for p in _adaylar if os.path.exists(p)), None)
assert src, ("Yamali sifirdan.py yok. Depodaki surum YAMASIZ; sunlardan biri "
             "Drive'da bulunmali: " + " | ".join(_adaylar))
shutil.copy(src, f"{KOD}/sifirdan.py")
print("  sifirdan.py <-", src)
KOD_MD5 = hashlib.md5(open(f"{KOD}/sifirdan.py", "rb").read()).hexdigest()
print("  calisacak sifirdan.py md5 =", KOD_MD5[:10])
if src != f"{EV}/kod/sifirdan.py":                     # bu deneye kendi kopyasi
    shutil.copy(src, f"{EV}/kod/sifirdan.py.tmp")
    os.replace(f"{EV}/kod/sifirdan.py.tmp", f"{EV}/kod/sifirdan.py")
open(f"{EV}/kod_md5.txt", "w").write(KOD_MD5 + "  " + src)   # CLAUDE.md 9

# --- DENETIM 1: DOSYADA yama var mi --------------------------------------
_k = open(f"{KOD}/sifirdan.py", encoding="utf-8").read()
for g in ('MASK_KEY = os.environ.get("MASK_KEY"',
          'self.mask_key = None',
          'm[:, self.mask_key] = True'):
    assert g in _k, f"sifirdan.py YAMASIZ (eksik: {g[:34]}...)"
print("DENETIM 1 GECTI   dosyada maske yamasi var")

# --- DENETIM 2: DEFTERE IMPORT EDILEN modul de yamali mi? ----------------
# 13 Eylul arizasi tam buradaydi: egitim yamali klondan kosuyordu, defter
# BASKA bir klondan import ediyordu. blk.mask_key = 1 olu oznitelik yaratti,
# hata vermedi, maskeyle egitilmis modeli maskesiz olctu.
for _v in ("MASK_KEY", "MASK_BLK", "RESUME_FROM", "INIT_FROM", "OUT"):
    os.environ.pop(_v, None)                   # analiz modulu MASKESIZ import edilir
os.environ.update(PRESET="grok_uzun", ARMS="A", SEEDS="0",
                  HOP2_FRAC="0.10", MEM_AT="4")
sys.path.insert(0, KOD)
sys.modules.pop("sifirdan", None)
import sifirdan as S

print("  S.__file__ =", S.__file__)
assert os.path.realpath(S.__file__).startswith(os.path.realpath(KOD)), \
    f"YANLIS MODUL yuklendi: {S.__file__}"
assert "self.mask_key" in inspect.getsource(S.Block.forward), \
    "Yuklenen modulun Block.forward'i maskeyi OKUMUYOR"
print("DENETIM 2 GECTI   import edilen modul yamali")

# --- DENETIM 3: maske etkili mi + kapatinca BIT-AYNI mi? -----------------
_n = S.Net("A", S.CFG).to(S.DEV).eval()
_x = torch.randint(0, _n.emb.num_embeddings, (8, S.T_LEN), device=S.DEV)
with torch.no_grad():
    _a = _n(_x)[0].float().clone()
    for _b in _n.blocks: _b.mask_key = 1
    _m = _n(_x)[0].float().clone()
    for _b in _n.blocks: _b.mask_key = None
    _c = _n(_x)[0].float().clone()
assert torch.equal(_a, _c), "maske kapatilinca eski hale donmuyor (no-op DEGIL)"
assert float((_a - _m).abs().max()) > 1e-3, "MASKE ETKISIZ -> olcum bozuk olurdu"
print(f"DENETIM 3 GECTI   maske etkili (fark {float((_a-_m).abs().max()):.1f}), "
      f"kapatinca bit-ayni")
del _n, _x, _a, _m, _c

def snaplar(klasor, kol="A", tohum=0):
    """Anlik goruntuleri GLOB ile bul. Dosya adi bicimini VARSAYMA:
    snap_A_s0_070000.pt SIFIR DOLGULU; elle format kurmak hata kaynagi."""
    d = {}
    for p in glob.glob(f"{klasor}/snap_{kol}_s{tohum}_*.pt"):
        m = re.search(r"_(\d+)\.pt$", os.path.basename(p))
        if m: d[int(m.group(1))] = p
    return dict(sorted(d.items()))

TEMIZ = {k: os.environ[k] for k in
         ("PATH", "HOME", "LANG", "LD_LIBRARY_PATH", "CUDA_VISIBLE_DEVICES")
         if k in os.environ}          # alt surece SIZINTISIZ ortam
print("\nKURULUM TAMAM")


In [ ]:
# ============================================================================
# HUCRE 2 - OLCUM HATTI + DENETIM 4 (D3 BASLAMADAN ONCE)
# CLAUDE.md 4: tek olcum yolu. CLAUDE.md 10a: bilinen degere karsi sina.
# D3'un sayisi, D'nin sayisiyla kiyaslanacak -> IKISI DE BU HATTAN gecmeli.
# Sabit 0.322'ye guvenmiyoruz; D'yi bu defter YENIDEN olcer.
# ============================================================================
facts, pairs, one, tr2, comp_, ent_ev, seen_ent, unseen_ent, ent2_ev = S.build_data()
lo, hi = S.ENT_OFF, S.ENT_OFF + S.CFG["N_ENT"]
N = 3000

SET = {}
for ad, lst in (("COMP", comp_), ("ENT", ent_ev), ("ENT2", ent2_ev)):
    l = lst[:N]                       # (e, r1, r2, b, a)
    E = S.enc_two(l)
    SET[ad] = dict(X=E[0], poz=E[1],
                   gold=(S.ENT_OFF + np.array([x[4] for x in l])).astype(np.int64),
                   kis=(S.ENT_OFF + np.array([facts[x[0], x[2]] for x in l])).astype(np.int64))
_rg = np.random.RandomState(7)
_e = _rg.randint(S.CFG["N_ENT"], size=N); _r = _rg.randint(S.CFG["N_REL"], size=N)
_E1 = S.enc_one([(int(a), int(b), int(facts[a, b])) for a, b in zip(_e, _r)])
SET["1hop"] = dict(X=_E1[0], poz=_E1[1], gold=_E1[2], kis=None)
print("olcum kumeleri:", {k: len(v["X"]) for k, v in SET.items()})

_net = S.Net("A", S.CFG).to(S.DEV)

def maske_kur(net, blok):
    for i, b in enumerate(net.blocks): b.mask_key = 1 if i in blok else None

@torch.no_grad()
def OLC(net, bs=500):
    o = {}
    for ad, d in SET.items():
        X, poz, g = d["X"], d["poz"], d["gold"]; ok, kk = [], []
        for i in range(0, len(X), bs):
            xb = torch.from_numpy(X[i:i+bs]).to(S.DEV)
            ix = torch.from_numpy(poz[i:i+bs]).to(S.DEV)
            ar = torch.arange(len(ix), device=S.DEV); lg, _ = net(xb)
            p = (lg.float()[ar, ix][:, lo:hi].argmax(-1) + lo).cpu().numpy()
            ok.append(p == g[i:i+bs])
            if d["kis"] is not None: kk.append(p == d["kis"][i:i+bs])
        o[ad] = float(np.concatenate(ok).mean())
        if kk: o[ad + "_kis"] = float(np.concatenate(kk).mean())
    return o

def olc_snap(yol, blok):
    sd = torch.load(yol, map_location=S.DEV)
    _net.load_state_dict({k: v.float() for k, v in sd.items()})
    maske_kur(_net, blok); _net.eval(); r = OLC(_net); maske_kur(_net, ()); return r

def olc_ortalama(yollar, blok):
    acc = None
    for y in yollar:
        sd = torch.load(y, map_location=S.DEV)
        if acc is None: acc = {k: v.float().clone() for k, v in sd.items()}
        else:
            for k in acc: acc[k] += sd[k].float()
    for k in acc: acc[k] /= len(yollar)
    _net.load_state_dict(acc); maske_kur(_net, blok); _net.eval()
    r = OLC(_net); maske_kur(_net, ()); return r

# --- DENETIM 4: D'nin BILINEN egri degerini yeniden uretebiliyor muyuz? --
# Tutuyorsa: veri bolmesi + konfig + maske + olcum yolu, D ile AYNI hizada.
# Tutmuyorsa: bu defterle D3'u D ile kiyaslamak ANLAMSIZ -> devam etme.
_dk, _dm = REF["D"]
_eg = f"{_dk}/egri_A_s0.json"
assert os.path.exists(_eg), f"D'nin egrisi yok: {_eg} (deney7 Drive'da mi?)"
_E = {r["step"]: r for r in json.load(open(_eg))}
_sn = snaplar(_dk)
_ad = [a for a in (70000, 75000, 65000) if a in _sn and a in _E][:1]
assert _ad, "D'nin 65-75 bin anlik goruntusu bulunamadi"
_a = _ad[0]
_ml = olc_snap(_sn[_a], _dm)          # maske ACIK  (D'nin egitim yapilandirmasi)
_mk = olc_snap(_sn[_a], ())           # maske KAPALI (yanlis yapilandirma - karsilastirma)
_fark = abs(_ml["ENT"] - _E[_a]["ent"])
print(f"\nDENETIM 4  D @ {_a}   egri ent = {_E[_a]['ent']:.3f}")
print(f"   maske ACIK  -> ENT {_ml['ENT']:.3f}  (fark {_fark:.3f})   <- dogru olan")
print(f"   maske KAPALI-> ENT {_mk['ENT']:.3f}  (fark {abs(_mk['ENT']-_E[_a]['ent']):.3f})")
assert _fark < 0.03, ("Olcum hatti D'nin egrisini yeniden uretemiyor. "
                      "Veri bolmesi / konfig / offsetler uyusmuyor - kiyas ANLAMSIZ.")
print("DENETIM 4 GECTI   olcum hatti D ile ayni hizada (veri+konfig+offset)")

# --- DENETIM 5: EGITIMIN MASKESI -> davranistan DEGIL, KAYITLI KONFIGDEN --
# DENETIM 4 maskeyi AYIRT EDEMEZ: D'de maskeli 0.157 / maskesiz 0.150, ikisi de
# egrinin 0.03 yakininda. Cunku D kisayolu zaten kullanmiyor. Davranissal
# kontrol bu isi goremez; surdur_*.pt icindeki cfg gorur.
_sp = glob.glob(f"{_dk}/surdur_*.pt")
assert _sp, "D'nin surdurme paketi yok - konfig kapisi SINANAMIYOR"
_c = torch.load(_sp[0], map_location="cpu", weights_only=False).get("cfg", {})
assert _c.get("MASK_KEY") == "1" and tuple(_c.get("MASK_BLK") or ()) == _dm, \
    f"Konfig kapisi bozuk: MASK_KEY={_c.get('MASK_KEY')!r} MASK_BLK={_c.get('MASK_BLK')!r}"
print(f"DENETIM 5 GECTI   kayitli-konfig kapisi calisiyor "
      f"(D: MASK_KEY={_c['MASK_KEY']!r} MASK_BLK={_c['MASK_BLK']})")


In [ ]:
# ============================================================================
# HUCRE 3 - REFERANSLARI BU HATTAN OLC (D ve A, 60-80 bin penceresi)
# Sabit sayi (0.322) kullanmiyoruz: kiyasin iki tarafi ayni koddan gecsin.
# ============================================================================
PENCERE = [60000, 65000, 70000, 75000, 80000]
REFERANS = {}
for ad, (kl, mk) in REF.items():
    sn = snaplar(kl)
    var = [a for a in PENCERE if a in sn]
    if len(var) < 5:
        print(f"{ad}: {len(var)}/5 nokta -> atlandi ({kl})"); continue
    tek = [olc_snap(sn[a], mk)["ENT"] for a in var]
    o = olc_ortalama([sn[a] for a in var], mk)
    REFERANS[ad] = dict(tek=tek, ort=o, maske=list(mk))
    print(f"{ad:>3s} (maske {mk or 'yok'}): tek {[f'{v:.3f}' for v in tek]}  "
          f"ORT ENT {o['ENT']:.3f}  kis {o['ENT_kis']:.3f}  COMP {o['COMP']:.3f}  "
          f"ENT2 {o['ENT2']:.3f}")

REF_D = REFERANS["D"]["ort"]["ENT"]
print(f"\nREF_D (bu defterin olcumu) = {REF_D:.3f}")
print(f"onceki defterin olcumu      = 0.322   fark {abs(REF_D-0.322):.3f}")
if abs(REF_D - 0.322) > 0.02:
    print("!! FARK BUYUK - iki defterin olcum kumeleri ayni degil.")
    print("   Hukum BU defterin REF_D'sine gore verilir; belgeye bu not dusulur.")
json.dump({k: {"tek": v["tek"], "ort": v["ort"], "maske": v["maske"]}
           for k, v in REFERANS.items()}, open(f"{OUT}/referans_D_A.json", "w"), indent=1)
os.system(f"cp -f {OUT}/referans_D_A.json {EV}/ 2>/dev/null")


In [ ]:
# ============================================================================
# HUCRE 4 - D3'U BASLAT   (~85 dk, tek kol)
# Kopma olursa bu hucreyi tekrar calistir: biten atlanir, yarim kalan surdurur.
# ============================================================================
ORTAK = dict(PRESET="grok_uzun", STEPS="120000", ARMS="A", SEEDS="0",
             HOP2_FRAC="0.10", MEM_AT="4", RESUME_EVERY="1")
KOSULAR = [("D3", {"MASK_KEY": "1",
                   "MASK_BLK": ",".join(str(b) for b in MASKE_D3)})]
print("kosulacak:", KOSULAR)

open("/content/kos9.py", "w").write(f"""
import os, sys, subprocess, time
TEMIZ = {TEMIZ!r}
for ad, ek in {KOSULAR!r}:
    d = "{OUT}/" + ad
    os.makedirs(d, exist_ok=True)
    if os.path.exists(d + "/verdict.json"):
        print(f"[{{ad}}] zaten bitmis, atlandi", flush=True); continue
    sur = d + "/surdur_A_s0.pt"
    env = dict(TEMIZ, OUT=d, **{ORTAK!r}, **ek,
               RESUME_FROM=(sur if os.path.exists(sur) else ""))
    print(f"[{{ad}}] basliyor {{ek}}" + ("  [SURDURULUYOR]" if os.path.exists(sur) else ""),
          flush=True)
    t = time.time()
    r = subprocess.run([sys.executable, "-u", "sifirdan.py"], cwd="{KOD}", env=env,
                       stdout=open(f"/content/log_{{ad}}.txt", "a"), stderr=subprocess.STDOUT)
    print(f"[{{ad}}] bitti rc={{r.returncode}}  {{(time.time()-t)/60:.1f}} dk", flush=True)
    if r.returncode != 0:
        print(f"[{{ad}}] HATA -> log_{{ad}}.txt", flush=True); break
print("BITTI", flush=True)
""")
p = subprocess.Popen([sys.executable, "-u", "/content/kos9.py"],
                     stdout=open("/content/surucu9.txt", "w"), stderr=subprocess.STDOUT)

# Yedekleme. surdur_*.pt ~102 MB ve HER OLCUMDE yeniden yazilir: duz cp
# ortasinda runtime olurse Drive'da YARIM dosya kalir, /content ise silinmistir.
# CLAUDE.md 7 -> buyuk dosya .tmp + mv ile. Nabiz dosyasi; ps|grep KULLANMA.
_yed = """while true; do
  ( cd __OUT__ && find . -type f ! -name 'surdur_*.pt' -print0 |
      xargs -0 -I@ cp -u --parents @ __EV__/ ) 2>/dev/null
  for f in __OUT__/*/surdur_*.pt; do
    [ -e "$f" ] || continue
    rel="${f#__OUT__/}"
    mkdir -p "__EV__/$(dirname "$rel")"
    cp -f "$f" "__EV__/$rel.tmp" && mv -f "__EV__/$rel.tmp" "__EV__/$rel"
  done
  cp -f /content/log_*.txt /content/surucu9.txt __EV__/ 2>/dev/null
  touch /content/yedek_nabiz9
  sleep 300
done
"""
open("/content/yedek9.sh", "w").write(
    _yed.replace("__OUT__", OUT).replace("__EV__", EV))
subprocess.Popen(["bash", "/content/yedek9.sh"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f"pid {p.pid}   ~85 dk   yedek: {EV} (5 dk)")


In [ ]:
# ============================================================================
# HUCRE 5 - DURUM + EGITIMIN MASKESI GERCEKTEN ACIK MI
# ============================================================================
import os, re, time
print(open("/content/surucu9.txt").read() if os.path.exists("/content/surucu9.txt") else "")
nb = "/content/yedek_nabiz9"
print("nabiz:", f"{time.time()-os.path.getmtime(nb):.0f} sn once"
      if os.path.exists(nb) else "yok")
y = "/content/log_D3.txt"
if os.path.exists(y):
    L = open(y).read().splitlines()
    for l in [x.rstrip()[:140] for x in L if re.match(r"^\s+\d+\s+loss", x)][-3:]: print("  ", l)
    for l in [x.rstrip()[:100] for x in L if re.match(r"^\s+adim\s+\d+", x)][-1:]: print("  ", l)

sn = snaplar(f"{OUT}/D3"); print("\nanlik goruntuler:", list(sn.keys())[-6:])

# --- EGITIM MASKESI: KAYITLI KONFIGDEN (hukum veren kontrol) -------------
_sp = glob.glob(f"{OUT}/D3/surdur_*.pt")
if _sp:
    _c = torch.load(_sp[0], map_location="cpu", weights_only=False).get("cfg", {})
    print(f"\nEGITIM KONFIGI  MASK_KEY={_c.get('MASK_KEY')!r}  "
          f"MASK_BLK={_c.get('MASK_BLK')!r}")
    assert _c.get("MASK_KEY") == "1" and tuple(_c.get("MASK_BLK") or ()) == MASKE_D3, \
        "EGITIM YANLIS MASKEYLE KOSUYOR -> DURDUR"
    print("   TAMAM: egitim D3 maskesiyle kosuyor")
else:
    print("\n(surdurme paketi henuz yok - ilk olcumden sonra kontrol edilecek)")

# --- ERKEN UYARI: D3'u D ile AYNI ADIMDA kiyasla, ta basindan ----------
# Amac: yanlisi 120.000'de degil, 20.000'de gormek. Bunlar UYARIDIR;
# birincil hukum yine 60-80 bin penceresinden verilir (ONKAYIT, degismez).
# Esikler kosudan ONCE yazildi:
#   SAGLIK   comp(D3)  >= comp(D) - 0.10      2 ardisik noktada ihlal -> bak
#   DOLANMA  kisayol(D3) <= 0.15              maske dolanilmis olabilir
#   YOL      ent(D3)   >= ent(D)  - 0.05      yetenek hic gelismiyorsa
_e3 = f"{OUT}/D3/egri_A_s0.json"
_ed = f"{REF['D'][0]}/egri_A_s0.json"
if os.path.exists(_e3) and os.path.exists(_ed):
    E3 = {r["step"]: r for r in json.load(open(_e3))}
    ED = {r["step"]: r for r in json.load(open(_ed))}
    ort = [s for s in sorted(E3) if s in ED]
    print(f"\nERKEN UYARI  (esikler kosudan once yazildi)")
    print(f"{'adim':>7s} {'compD3':>7s} {'compD':>6s} {'d':>7s} | {'entD3':>6s} "
          f"{'entD':>6s} | {'kisD3':>6s} {'kisD':>6s} | durum")
    _ihlal = []
    for s_ in ort[-8:]:
        a, b = E3[s_], ED[s_]
        dc = a["comp"] - b["comp"]
        u = []
        if dc < -0.10: u.append("SAGLIK")
        if a["ent_shortcut"] > 0.15: u.append("DOLANMA")
        if a["ent"] < b["ent"] - 0.05: u.append("YOL")
        if u: _ihlal.append((s_, u))
        print(f"{s_:7d} {a['comp']:7.3f} {b['comp']:6.3f} {dc:+7.3f} | "
              f"{a['ent']:6.3f} {b['ent']:6.3f} | {a['ent_shortcut']:6.3f} "
              f"{b['ent_shortcut']:6.3f} | {','.join(u) if u else 'tamam'}")
    _ard = [x for k, x in enumerate(_ihlal)
            if k and _ihlal[k-1][0] in ort and ort.index(x[0]) - ort.index(_ihlal[k-1][0]) == 1]
    print("  ->", f"{len(_ihlal)} noktada uyari, {len(_ard)} tanesi ardisik"
          if _ihlal else "butun kontrollerden gecti")
    if _ard:
        print("  !! ARDISIK IHLAL: kosuyu kesmeyi dusun, sonuca kadar bekleme")
    _p = [s_ for s_ in (60000, 65000, 70000, 75000, 80000) if s_ in E3]
    print(f"  birincil okuma icin gereken 5 nokta: {len(_p)}/5 hazir  "
          f"(80.000'de okunabilir, ~57 dk)")

# --- davranissal kiyas: BILGI AMACLI, hukum vermez ----------------------
_eg = f"{OUT}/D3/egri_A_s0.json"
if sn and os.path.exists(_eg):
    E = {r["step"]: r for r in json.load(open(_eg))}
    a = max(k for k in sn if k in E)
    print(f"  bilgi: adim {a}  egri {E[a]['ent']:.3f} | maskeli "
          f"{olc_snap(sn[a], MASKE_D3)['ENT']:.3f} | maskesiz "
          f"{olc_snap(sn[a], ())['ENT']:.3f}")


In [ ]:
# ============================================================================
# HUCRE 6 - ONCEDEN KAYITLI OKUMA   60.000-80.000, 5 nokta
# Kural ONKAYIT_D3_ZEMIN_KAT.md'de kosudan once yazildi. DEGISTIRILMEZ.
# ============================================================================
BAND = 0.05
sn = snaplar(f"{OUT}/D3")
var = [a for a in PENCERE if a in sn]
if len(var) < 5:
    print(f"{len(var)}/5 nokta hazir {var} -> HENUZ ERKEN (5 nokta sart)")
else:
    tek = [olc_snap(sn[a], MASKE_D3)["ENT"] for a in var]
    o = olc_ortalama([sn[a] for a in var], MASKE_D3)
    v, dD = o["ENT"], o["ENT"] - REF_D
    hkm = ("DERINLIK ONEMSIZ (esdegerlik)" if abs(dD) < BAND else
           "GENIS DAHA IYI" if dD >= BAND else "GENIS DAHA KOTU")
    print(f"D3 tek noktalar {[f'{x:.3f}' for x in tek]}   en iyi {max(tek):.3f}")
    print(f"D3 ORTALAMA  ENT {v:.3f}  kisayol {o['ENT_kis']:.3f}  COMP {o['COMP']:.3f}  "
          f"ENT2 {o['ENT2']:.3f}  1hop {o['1hop']:.3f}")
    print(f"referans  D {REF_D:.3f}  (ayni hattan)   A "
          f"{REFERANS.get('A', {}).get('ort', {}).get('ENT', float('nan')):.3f}")
    print(f"\nBIRINCIL HUKUM: {hkm}   (D3 - D = {dD:+.3f}, band +/-{BAND})")
    e2 = o["ENT2"] - REFERANS["D"]["ort"]["ENT2"]
    print(f"OZGULLUK: ENT2 farki {e2:+.3f}  -> "
          f"{'TAMAM' if e2 < 0.05 else 'IHLAL - sonuc gecersiz'}")
    print(f"SAGLIK  : COMP {o['COMP']:.3f} (>=0.80)  1hop {o['1hop']:.3f} (>=0.98)")
    print(f"DOLANMA : kisayol {o['ENT_kis']:.3f}  (D: {REFERANS['D']['ort']['ENT_kis']:.3f})")
    json.dump(dict(pencere=var, tek=tek, ortalama=o, REF_D=REF_D, hukum=hkm,
                   maske=list(MASKE_D3)), open(f"{OUT}/onkayit_D3.json", "w"), indent=1)
    os.system(f"cp -f {OUT}/onkayit_D3.json {EV}/ 2>/dev/null")


In [ ]:
# ============================================================================
# HUCRE 7 - KURTARMA BETIGI -> Drive  (defter kaybolursa TEK hucre yeter)
# ============================================================================
_mb = ",".join(str(b) for b in MASKE_D3)
open(f"{EV}/kod/kurtar.py", "w").write(f'''
# !python /content/drive/MyDrive/deney9_D3/kod/kurtar.py
import os, sys, subprocess, shutil
EV, OUT, KOD = "{EV}", "{OUT}", "{KOD}"
os.makedirs(OUT, exist_ok=True)
subprocess.run(["bash","-lc", "cp -ru " + EV + "/. " + OUT + "/ 2>/dev/null; "
                "rm -rf " + KOD + "; git clone -q "
                "https://github.com/sekerahmet/sekerai.git " + KOD])
src = EV + "/kod/sifirdan.py"
if os.path.exists(src): shutil.copy(src, KOD + "/sifirdan.py")
k = open(KOD + "/sifirdan.py", encoding="utf-8").read()
for g in ('MASK_KEY = os.environ.get("MASK_KEY"', "self.mask_key = None",
          "m[:, self.mask_key] = True"):
    assert g in k, "sifirdan.py YAMASIZ: " + g[:30]
print("yama dogrulandi")
d = OUT + "/D3"; os.makedirs(d, exist_ok=True)
if os.path.exists(d + "/verdict.json"):
    print("D3 zaten bitmis"); sys.exit()
sur = d + "/surdur_A_s0.pt"
env = dict(os.environ, OUT=d, PRESET="grok_uzun", STEPS="120000", ARMS="A",
           SEEDS="0", HOP2_FRAC="0.10", MEM_AT="4", RESUME_EVERY="1",
           MASK_KEY="1", MASK_BLK="{_mb}",
           RESUME_FROM=(sur if os.path.exists(sur) else ""))
print("basliyor" + ("  [SURDURULUYOR]" if os.path.exists(sur) else ""))
subprocess.run([sys.executable, "-u", "sifirdan.py"], cwd=KOD, env=env)
''')
print("kurtar.py yazildi, maske =", _mb)


---

## Hakemlik notlari (kosudan once)

**K1.** Birincil kiyasin iki tarafi (D ve D3) **ayni olcum hattindan** gecer.
Sabit `0.322` kullanilmaz; D bu defterde yeniden olculur (Hucre 3) ve eski
degerle farki basilir.

**K2.** Olcum hatti, D3 baslamadan **once** D'nin bilinen egrisine karsi
sinanir (DENETIM 4). Tutmazsa assert patlar ve 85 dakika harcanmaz.

**K3.** Maske sabiti `MASKE_D3` **tek yerde** tanimli; kurtarma betigi ve
launcher ondan turetir. Elle kopyalanan sabit yok.

**K4.** Egitimin maskesi **kayitli konfigden** dogrulanir (Hucre 5), davranistan
degil. Smoke testte gorundu ki davranissal kontrol bu isi GOREMIYOR: D'de
maskeli 0.157 / maskesiz 0.150, ikisi de egrinin (0.145) 0.03 yakininda.
`surdur_*.pt` icindeki `cfg["MASK_BLK"]` ise tartismasiz soyluyor. Kapinin
kendisi de D/K/A uzerinde sinandi (D=(6,7), K key=2, A=None).

**K6.** Smoke test yapildi: Hucre 2 ve 3'un **birebir metni** (md5 dogrulamali)
gercek veriyle kosturuldu. D = 0.322, A = 0.062 uretti; onceki defterin
sayilariyla farki 0.000.

**K7.** Yedekleme, 102 MB'lik `surdur_*.pt` icin `.tmp` + `mv` kullanir; duz
`cp` ortasinda runtime olurse Drive'da yarim dosya kalirdi (CLAUDE.md 7).

**K8.** Depodaki `sifirdan.py` YAMASIZ. Kaynak sirayla aranir, bulunamazsa
acik mesajla durulur; kullanilan surumun md5'i `kod_md5.txt`e yazilir.

**K9.** Egitim yolu smoke testi yapildi (gercek GPU): PRESET=grok_uzun,
STEPS=200, MASK_BLK=1..7, COMPILE acik -> rc=0, 111 sn, anlik goruntuler
sifir dolgulu, kayitli cfg MASK_BLK=(1,2,3,4,5,6,7). Hatali girdiler
(`1,8` ve bos) assert atesledi.

**K10.** Esdegerlik bandi +/-0.05, D ile E1'in FARKLI TOHUM farkindan (0.016)
turetildi; ama D3-D kiyasi AYNI TOHUM. Ayni tohumda gurultu daha az, yani
band fazla genis ve **esdegerlik ilan etmeye egilimli** - bizim bekledigimiz
sonuc yonunde. Band onceden yazildi, degistirilmiyor; bu yanlilik raporda
belirtilecek.

**K5.** Bilinen ve KAPATILMAYAN eksikler: surdurme `MASK_KEY`'i dogrulamiyor
(kurtar.py dogru maskeyi verdigi icin bu defterde risk dusuk); ornek-basina
tablo surdurmede sifirlaniyor.

## Sonuca gore (onceden yazildi)

| hukum | ne yapilir |
|---|---|
| esdegerlik | derinlik onemsiz -> **Adim 2** acilir |
| genis daha iyi | agirlik-ortalamasi bulgusu D3 ile yeniden okunur |
| genis daha kotu | derinlik onemli -> **Adim 2 kapali kalir** |
